# Generating the engineer data for SkillMatch

A company's engineering bench is private HR data, so there is no public dataset of engineers
rated on seniority, availability and the rest. I generate a synthetic bench instead. The values
are not all random: I build in a few correlations so the engineers behave like real people,
each one explained in the section that sets it.

## 1. Setup

In [2]:
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker()         
fake_in = Faker("en_IN") 
Faker.seed(42)
np.random.seed(42)

N = 3000  # how many engineers to make

## 2. Seniority, experience and domain

Seniority is the trait everything else keys off. Engineers with higher seniority tend to have
more years behind them, and more years usually means a deeper grasp of their domain.

In [3]:
# seniority is roughly a bell curve
seniority = np.clip(np.random.normal(50, 20, N), 0, 99)

# more senior people have more years of experience
years = np.clip(seniority / 99 * 22 + np.random.normal(0, 2, N), 0, 30)

# more years usually means deeper domain knowledge
domain = np.clip(30 + years * 1.2 + np.random.normal(0, 14, N), 0, 99)

## 3. Client communication

More senior engineers usually spend more time in front of clients, so communication scores rise
with seniority.

In [4]:
communication = np.clip(0.5 * seniority + np.random.normal(25, 14, N), 0, 99)

## 4. Region and timezone overlap

Timezone overlap is scored against a US head office, so it comes down to where the engineer is
based.

In [5]:
regions = np.random.choice(["Americas", "EMEA", "APAC"], N)

# how much each region overlaps with US working hours
region_overlap = {"Americas": 82, "EMEA": 55, "APAC": 28}
timezone_base = np.array([region_overlap[r] for r in regions])
timezone = np.clip(timezone_base + np.random.normal(0, 10, N), 0, 99)

## 5. Bandwidth

Senior engineers are often spread across several projects at once, which leaves them with
slightly less free bandwidth.

In [6]:
bandwidth = np.clip(np.random.normal(65, 20, N) - seniority / 99 * 18, 0, 99)

## 6. Current availability

This is separate from bandwidth. Bandwidth is how much time an engineer generally likes to
commit; availability is whether they are already tied up on another project right now. Senior
people are a little more likely to be booked, since they get pulled onto more work.

In [7]:

# a rough "how busy right now" roll, nudged up a bit for more senior people
busy_score = seniority / 99
roll = np.random.uniform(0, 1, N) + busy_score * 0.3

availability_status = np.where(
    roll > 1.15, "Fully booked",
    np.where(roll > 0.75, "Partially committed", "Available"),
)

# how many weeks until they are free. 0 if already available.
available_in_weeks = np.zeros(N, dtype=int)
partial = availability_status == "Partially committed"
booked = availability_status == "Fully booked"
available_in_weeks[partial] = np.random.randint(1, 5, partial.sum())
available_in_weeks[booked] = np.random.randint(4, 17, booked.sum())

pd.Series(availability_status).value_counts()

Available              1753
Partially committed    1167
Fully booked             80
Name: count, dtype: int64

## 7. Discipline, vertical and names

Discipline is the team an engineer sits in, with backend and frontend the most common and the
others rarer. Names are 70% Indian and 30% international, assigned at random per row.

In [8]:
disciplines = np.random.choice(
    ["Frontend", "Backend", "Data / Database", "DevOps", "Cloud", "Networking", "AI / ML", "Other"],
    N,
    p=[0.18, 0.22, 0.13, 0.12, 0.11, 0.08, 0.10, 0.06],
)

verticals = np.random.choice(
    ["FinTech", "Healthcare", "E-commerce", "SaaS", "Gaming", "Media",
     "Logistics", "EdTech", "Telecom", "Government", "Other"], N)

# 70% Indian names, 30% international, assigned randomly (not tied to region/vertical)
is_indian = np.random.rand(N) < 0.7
names = [fake_in.name() if is_indian[i] else fake.name() for i in range(N)]

## 8. Put it together and save

In [9]:
engineers = pd.DataFrame({
    "name": names,
    "discipline": disciplines,
    "region": regions,
    "vertical": verticals,
    "years_experience": np.round(years).astype(int),
    "seniority": np.round(seniority).astype(int),
    "domain": np.round(domain).astype(int),
    "communication": np.round(communication).astype(int),
    "timezone": np.round(timezone).astype(int),
    "bandwidth": np.round(bandwidth).astype(int),
    "availability_status": availability_status,
    "available_in_weeks": available_in_weeks,
})

engineers.to_csv("engineers.csv", index=False)
print("saved engineers.csv with", len(engineers), "engineers")
engineers.head()

saved engineers.csv with 3000 engineers


,name,discipline,region,vertical,years_experience,seniority,domain,communication,timezone,bandwidth,availability_status,available_in_weeks
0,Allison Hill,Frontend,EMEA,Media,10,60,26,66,49,80,Available,0
1,Liam Chaudry,Backend,Americas,Logistics,9,47,32,64,85,31,Partially committed,2
2,Pahal Balay,Cloud,Americas,FinTech,13,63,33,63,86,98,Available,0
3,Girik Kamdar,Backend,EMEA,FinTech,22,80,48,38,49,44,Available,0
4,Lance Hoffman,DevOps,APAC,SaaS,11,45,40,45,40,56,Partially committed,3
